# Generate masks

**What it does.** Turn raw microscope images into per-object masks for cells, nuclei and pathogens.

**When to use it.** First step of almost every spaCR analysis. Run it once per plate, before anything that needs objects rather than pixels.

**What you get.** A `masks/` folder beside the source images, and a measurement database seeded with the objects it found.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.core.preprocess_generate_masks`

```
preprocess_generate_masks(settings)
```

Turn a folder of raw microscopy images into per-channel Cellpose masks ready for :func:`spacr.measure.measure_crop`.

In [ ]:
from spacr.core import preprocess_generate_masks

## 3. Settings

`spacr.settings.set_default_settings_preprocess_generate_masks` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import set_default_settings_preprocess_generate_masks

defaults = set_default_settings_preprocess_generate_masks({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (bool) - After segmentation, rewrite the cell masks so labels
    # split across a single pathogen or nucleus are merged, and cell
    # fragments with no nucleus are absorbed into the neighbour they
    # share most perimeter with. Needs cell, nucleus and pathogen
    # channels and is skipped for timelapse runs. Enable when large
    # infected cells come back fragmented. Default False.
    'adjust_cells': False,

    # (float or None) - The ratio of the z step to the xy pixel size (dz
    # / dxy), which is what tells 'volumetric' mode how far apart two
    # planes really are. At the true value, objects separated by a few
    # planes stay separate; left at 1.0 on a confocal stack, where the z
    # step is routinely 3-10x the xy pixel, the segmenter reads a 5 um
    # gap as a 5 pixel gap and fuses everything along z into columns.
    # spaCR will not assume a value: leave this None and set
    # voxel_size_z_um / voxel_size_xy_um instead and it is derived, but
    # if neither is known a volumetric run stops rather than silently
    # picking 1.0. Measure reads it too: on a 3-D mask it sets the z
    # spacing for every regionprops and distance-transform call, so
    # without it the 'outside' ring is as many planes thick as it is
    # pixels wide. Default None.
    'anisotropy': None,

    # (int) - Streaming pipeline only (pipeline_style='v2'): how many
    # whole field stacks are loaded into RAM before one Cellpose batch
    # is segmented. Larger values keep the GPU busier and cut the number
    # of read passes over the plate, at a memory cost of roughly one
    # full field stack each. Ignored entirely by the v1 pipeline.
    # Default 8.
    'batch_fields': 8,

    # (int) - How many images are held and processed together in one
    # pass: field stacks during normalization and Cellpose segmentation,
    # crops per step during classifier training and activation maps.
    # Raising it speeds runs up but increases RAM/VRAM roughly linearly;
    # lower it on out-of-memory errors. Defaults: 50 for mask
    # generation, 64 for training.
    'batch_size': 50,

    # (float) - Cellpose cellprob_threshold: only pixels whose predicted
    # cell probability exceeds it are assigned to a mask. Raise it to
    # shrink outlines and drop faint or spurious cells; lower it to grow
    # outlines and recover dim ones. Valid range roughly -6 to 6,
    # default 0. Lower it first when whole cells are missing.
    'cell_CP_prob': 0,

    # (float) - Cellpose flow_threshold: the maximum allowed error
    # between a candidate mask's recomputed flows and the network's
    # predicted flows. Masks above it are discarded, so lowering it
    # strips ragged or implausible cells but also loses real ones;
    # raising it keeps more. Usable range about 0-3 (GUI allows -1 to
    # 3). Default 1.0.
    'cell_FT': 1.0,

    # (int) - Multiplied by cell_background to give the intensity the
    # normalisation ceiling must reach: spaCR walks the 98th to 99.5th
    # percentile of the cell channel and takes the first value at or
    # above that product as the upper anchor. Raise it to push the
    # ceiling higher and dim the normalised image; lower it to brighten
    # faint cells. Default 10.
    'cell_Signal_to_noise': 10,

    # (float) - Splitting threshold for cell_intensity_split: only cells
    # whose area exceeds this multiple of the median cell area in the
    # image (or cell_min_object_area, whichever is larger) are
    # watershed-split. Lower it toward 1.5 to cut borderline clumps,
    # raise it to split only obvious doublets. Ignored unless
    # cell_intensity_split is True. Default 2.0.
    'cell_area_multiplier': 2.0,

    # (int) - Background intensity of the cell channel in raw image
    # units. Pixels below it are zeroed when remove_background_cell is
    # True, and it is multiplied by cell_Signal_to_noise to set the
    # intensity the normalisation ceiling must reach. Set it from a
    # genuinely empty region; too high and dim cells are erased. Default
    # 100.
    'cell_background': 100,

    # (int or None) - Zero-indexed raw acquisition channel that Cellpose
    # segments into cell masks; it also selects which channel the
    # cell_background, cell_Signal_to_noise and remove_background_cell
    # settings are applied to during preprocessing. Set to None and no
    # cell masks, cell table or cell crops are produced. At least one of
    # cell/nucleus/pathogen/organelle_channel must be an integer or the
    # run aborts. Default None.
    'cell_channel': None,

    # (int or None) - Expected cell diameter in pixels. Cellpose 4
    # rescales the image by 30/diameter before segmenting, so setting it
    # makes objects land near the size CPSAM was trained on; leave it
    # None to segment at native scale. Set it when cells are much larger
    # or smaller than ~30 px and masks come back fragmented or merged.
    # spacr.diameter.estimate_diameters proposes a value from your own
    # fields. Default None.
    'cell_diameter': None,

    # (bool) - Merge touching cell labels when the mean intensity along
    # their shared boundary is at least as high as the interior
    # intensity of the dimmer of the two, meaning there is no real
    # membrane edge between them. Use it to repair cells Cellpose cut in
    # half. The comparison statistic is set by
    # cell_intensity_threshold_method. Default False.
    'cell_intensity_merge': False,

    # (int) - Percentile from 0 to 100 of the dimmer cell's interior
    # intensity used as the merge reference when
    # cell_intensity_threshold_method is 'percentile'. Raising it toward
    # 95 sets a higher bar for the shared boundary to clear, so fewer
    # pairs merge; lowering it merges more. Ignored when the method is
    # 'mean'. Default 75.
    'cell_intensity_percentile': 75,

    # (bool) - Split oversized cell labels by distance-transform
    # watershed before the merge and filter steps. Objects larger than
    # cell_area_multiplier times the median cell area are seeded at
    # local distance maxima cell_min_distance apart and cut. Despite the
    # name no intensity is used. Enable when several touching cells
    # share one label. Default False.
    'cell_intensity_split': False,

    # (str) - Reference statistic that cell_intensity_merge compares the
    # shared-boundary intensity against: 'mean' uses the mean interior
    # intensity of the dimmer of the two cells, 'percentile' uses its
    # cell_intensity_percentile instead. Any value other than 'mean' is
    # treated as 'percentile'. Choose 'percentile' with a high
    # percentile to make merging rarer. Default 'mean'.
    'cell_intensity_threshold_method': 'mean',

    # (int or None) - Maximum cell area in pixels^2; objects larger than
    # this are deleted after segmentation. Use it to discard clumps or
    # debris blobs that Cellpose labelled as one huge cell. It only
    # deletes, it never splits - use cell_intensity_split for that. 0 or
    # None disables the filter. Default 0.
    'cell_max_area': 0,

    # (int or None) - Drops the brightest cells per field: objects whose
    # mean intensity is above this percentile (0-100) of the per-image
    # distribution of cell mean intensities are removed. Lower it to
    # about 99 to strip saturated blobs and fluorescent debris.
    # Relative, not an absolute intensity. Use 100 to disable. Default
    # 100.
    'cell_max_intensity_percentile': 100,

    # (int) - Minimum cell area in pixels^2. Passed to Cellpose as
    # min_size so undersized masks are dropped during segmentation, then
    # re-applied afterwards to delete any object below it. Raise it to
    # clear debris and fragments; set it too high and genuine small
    # cells disappear. 0 disables. Default 0.
    'cell_min_area': 0,

    # (int) - Minimum separation in pixels between watershed seeds when
    # cell_intensity_split cuts an oversized cell; seeds are local
    # maxima of the distance transform. Raise it for fewer, larger
    # fragments and to stop one cell shattering, lower it to separate
    # tightly packed cells. Ignored unless cell_intensity_split is True.
    # Default 10.
    'cell_min_distance': 10,

    # (int) - Drops the dimmest cells per field: the mean intensities of
    # all surviving cells are pooled and objects below this percentile
    # (0-100) of that per-image distribution are removed. Being
    # relative, it always removes roughly this share of objects, however
    # bright the field. Use it to shed out-of-focus cells. 0 disables.
    # Default 0.
    'cell_min_intensity_percentile': 0,

    # (int) - Absolute pixel-area floor for splitting: the split
    # threshold is the larger of cell_area_multiplier times the median
    # cell area and this value, so cells at or below it are never cut.
    # Raise it to protect small cells in fields where the median area is
    # low. Ignored unless cell_intensity_split is True. Default 100.
    'cell_min_object_area': 100,

    # (str) - Which weights segment cells. Cellpose 4 ships exactly one
    # stock model, 'cpsam', so the only other value that means anything
    # is a path to a checkpoint you trained yourself in Train Cellpose -
    # that path is loaded as pretrained_model and honoured. The pre-SAM
    # names ('cyto', 'cyto2', 'cyto3', 'nuclei') are still accepted so
    # old settings files load, but they are mapped to 'cpsam' on the way
    # in and reported once: Cellpose 4 would have resolved them to cpsam
    # silently anyway. Note what Cellpose 4 does and does not still
    # read: diameter IS honoured (eval rescales the image by
    # 30/diameter), while model_type and diam_mean are
    # accepted-and-ignored with a 'not used in v4.0.1+' log line.
    # Default 'cpsam'.
    'cell_model_name': 'cpsam',

    # (float) - For each touching pair of cell labels, the shared
    # boundary length divided by the smaller object's perimeter; pairs
    # at or above this fraction are merged into one cell. Low values
    # such as 0.1 merge aggressively and can fuse true neighbours, high
    # values only rejoin pieces of the same cell. 0 disables perimeter
    # merging. Default 0.
    'cell_perimeter_fraction': 0,

    # (bool) - Delete every cell label touching any of the four image
    # edges before measurement. Removes partial cells whose area and
    # total intensity are truncated and would bias per-cell statistics,
    # at the cost of losing objects - a large cost in fields where cells
    # are big relative to the field. Default False.
    'cell_remove_border_objects': False,

    # (list of int) - Zero-indexed image channels kept in merged/*.npy
    # and measured by measure_crop; each entry produces its own
    # <object>_channel_<n>_* intensity columns. The list length fixes
    # where masks land, so cell/nucleus/pathogen_mask_dim must shift if
    # you change it. Preprocessing silently resets it to range(n) when
    # it does not match the number of channel folders found. Default
    # [0,1,2,3].
    'channels': [0, 1, 2, 3],

    # (str) - Matplotlib colormap applied to single-channel image
    # previews and to plate heatmaps. Perceptually uniform maps
    # ('viridis', 'inferno', 'magma') keep intensity differences honest;
    # 'gray' matches how the raw microscope data looks. Any registered
    # matplotlib name works, with an '_r' suffix to reverse it. Default
    # 'inferno' for image plots, 'viridis' for plate heatmaps.
    'cmap': 'inferno',

    # (str) - Legacy and currently inert: nothing in spaCR reads this
    # key. The two preprocessing defaults set it to 'lzw' and the GUI
    # offers lzw/zlib/none, but the live segmentation path writes masks
    # as uint16 .npy via np.save, so no mask TIFF is ever produced and
    # changing this setting changes nothing. The only functions that
    # take a compression argument, io.save_object_mask and
    # mask_io.save_mask (which hardcodes 'lzw'), are never called from
    # anywhere in the package, and the sequencing HDF5 writer uses
    # comp_type/comp_level instead. Leave it at its 'lzw' default.
    'compression': 'lzw',

    # (bool) - Before processing, recursively scan src for images and
    # copy them into a single <src>/consolidated folder, prefixing each
    # filename with its subfolder names so nothing collides; src is then
    # repointed there. Use it when one plate's images are split across
    # per-well or per-channel subfolders. Copies, so disk use roughly
    # doubles. Default False.
    'consolidate': False,

    # (str or None) - Python regex with named groups used to pull
    # metadata out of raw image filenames. With metadata_type 'custom'
    # the pattern actually compiled is '(<your regex>).<ext>' (<ext> =
    # the most common image extension found in src) and it is consumed
    # by _extract_filename_metadata, which needs the groups wellID,
    # fieldID and chanID; plateID is optional (falls back to the source
    # folder name) and timeID/sliceID are optional (absent means None).
    # A filename missing a required group is skipped with 'Could not
    # extract information from filename ... using provided regex', and a
    # filename the pattern does not match at all is skipped too. With
    # metadata_type 'auto' the regex is instead tried first to rename
    # files into Yokogawa form, where only wellID is mandatory, falling
    # back to automatic detection if it fails. Default None.
    'custom_regex': None,

    # (bool) - Legacy force-cleanup switch: when True it overrides
    # keep_intermediate and keep_original_images so stack/, masks/, the
    # numeric per-channel folders and the orig/ raw backup are all
    # removed once merged/ is built. Cleanup is already the default, so
    # this is only needed to beat those keep flags. Deletion is skipped
    # unless every field of view reached merged/. Default False.
    'delete_intermediate': False,

    # (bool) - Legacy denoising toggle for the mask pipeline: no code
    # reads this key, so it has no effect. To actually denoise, set the
    # per-object restore settings (cell_restore_type /
    # nucleus_restore_type / pathogen_restore_type) to 'denoise', which
    # routes segmentation through Cellpose's CellposeDenoiseModel.
    # Default False.
    'denoise': False,

    # (int) - How many fields spacr.diameter.estimate_diameters reads
    # before it proposes cell_diameter, nucleus_diameter and
    # pathogen_diameter from blob statistics instead of leaving you to
    # guess. Fields are taken on an even stride across the sorted plate,
    # so rows and columns are both represented rather than the first few
    # wells; each field costs about a second of CPU and loads neither
    # torch nor Cellpose. Raise it to 10-20 when wells vary a lot or the
    # proposal comes back at low confidence, drop it to 2-3 for a quick
    # look. Default 5.
    'diameter_estimate_n_fields': 5,

    # (bool) - Validate the settings against the data they point at,
    # print what the run would do, and stop before any compute. Mask
    # generation and measure-and-crop check that src exists and holds
    # the expected files, that every channel and mask-plane index is
    # inside the number of planes actually present, that each value has
    # the type the pipeline expects, and that the models, barcode CSVs
    # or measurements.db the app needs are on disk; each problem is
    # printed with a suggested fix, followed by a plan listing files
    # found, objects to be segmented or measured, and where output would
    # land. Nothing is written, no model is loaded and the GPU is never
    # touched, so a settings mistake costs seconds instead of a whole
    # run. The organize-and-stitch pipeline reuses the flag to list the
    # file moves it would make. Default False.
    'dry_run': False,

    # (int) - How many randomly chosen merged image stacks are rendered
    # as segmentation-overlay previews after mask generation (in
    # timelapse mode, per-channel panels instead). Raise it to check
    # outlines and normalization across more fields of view, at the cost
    # of render time and larger PDFs; 0 skips previews entirely. Default
    # 1.
    'examples_to_plot': 1,

    # (int) - Base figure size in inches; figures are built square as
    # figuresize x figuresize and font sizes are derived from it
    # (legend, axis labels and ticks at 0.75x, overlay text at 0.5x).
    # Raise it when text is unreadable at publication scale, lower it to
    # fit panels on screen. Default 10; cluster grids cap total width at
    # 200 inches.
    'figuresize': 10,

    # (bool) - Legacy switch for the old post-Cellpose cleanup pass,
    # which re-ran size/intensity/border filtering and logged
    # '_after_filtration' object counts to the database. The current
    # Cellpose-SAM segmentation path never reads it, so toggling it
    # changes nothing; use the per-object <object>_min_area,
    # <object>_max_area and <object>_perimeter_fraction settings
    # instead. Default False.
    'filter': False,

    # (int) - Playback rate of the per-channel movies written to
    # <src>/movies from timelapse .npy stacks, and only when timelapse
    # is True. Raise it to skim long acquisitions, lower it to inspect
    # individual frames. Affects the movies only - never tracking,
    # segmentation or measurements. Default 2.
    'fps': 2,

    # (float or None) - Seconds between consecutive timepoints, straight
    # off the acquisition settings. It converts the frame index into a
    # real time column in the tracks table and is what turns a
    # displacement per frame into a speed; no linking decision depends
    # on it, so a wrong value rescales reported velocities without
    # changing which objects were joined to which. Left None, spaCR
    # falls back to the motility module's seconds_per_frame rather than
    # becoming a second source of truth for the same number. Default
    # None.
    'frame_interval_s': None,

    # (bool) - Keep the intermediate stack/ and masks/ folders after the
    # merged/ arrays are built. Off by default: only merged/ is kept
    # (masks are embedded in merged and recorded in the database).
    'keep_intermediate': False,

    # (bool) - Streaming pipeline only (pipeline_style='v2'): write each
    # in-memory NPZ batch out under merged/_scratch/ instead of
    # discarding it, so the intermediate a failing run was working on
    # can be inspected. Costs the disk the streaming pipeline exists to
    # save, so turn it on only while diagnosing. Default False.
    'keep_npz': False,

    # (bool) - Keep the original raw input images (in orig/). Off by
    # default to save disk space; the pixel data lives in merged/.
    'keep_original_images': False,

    # (float) - Percentile of the non-zero pixels in each channel used
    # as the low anchor when rescaling that channel to 0-1; the high
    # anchor is chosen automatically between the 98th and 99.5th
    # percentile. Raise it to crush more dim background to black, lower
    # it to preserve faint signal. Valid 0-100, default 2.
    'lower_percentile': 2,

    # (int) - Objective magnification, used only to derive expected
    # object sizes: pixel diameter is 2*mag+80 for cells, 0.75*mag+45
    # for nuclei and mag for pathogens, with min/max area limits of
    # diameter^2/4 and diameter^2*10. Explicit cell_diameter,
    # nucleus_diameter or pathogen_diameter override it. Set it to the
    # objective actually used (10, 20, 40, 60). Default 20.
    'magnification': 20,

    # (bool) - Run Cellpose segmentation for every object channel you
    # defined (cell, nucleus, pathogen, organelle) and write label
    # stacks to masks/<object>_mask_stack. Set False to do preprocessing
    # only - build the normalized arrays now and segment later - but
    # Measure will then have nothing to quantify. Default True.
    'masks': True,

    # (float or None) - Fraction of failed items above which the run
    # aborts rather than finishing and reporting. 0.2 means 'stop once
    # more than a fifth of the fields have failed', on the grounds that
    # whatever is left is no longer the experiment. The ledger is
    # stamped into the artifact before the abort, so the evidence
    # survives. None (the default) never aborts on rate alone - every
    # failure is still counted and reported, and the artifact is still
    # marked partial. Default None.
    'max_failure_rate': None,

    # (bool) - Legacy option that merged two touching pathogen labels
    # into one when their shared boundary exceeded 66% of the smaller
    # object's perimeter, so a single PV split by Cellpose counted once.
    # The current Cellpose-SAM path ignores it - use
    # pathogen_perimeter_fraction instead. Default False.
    'merge_pathogens': False,

    # (str) - Which filename convention raw images are parsed with.
    # 'cellvoyager' (default) and 'cq1' use built-in regexes, 'custom'
    # uses custom_regex, and 'auto' first renames the whole folder into
    # Yokogawa naming (with custom_regex if given, else automatic
    # detection) before parsing. Choose wrong and
    # plate/well/field/channel are misread, so images land in the wrong
    # channel folders.
    'metadata_type': 'cellvoyager',

    # (bool) - Run the automated motility assay after segmentation: it
    # rebuilds per-object measurements from merged/*.npy, cleans tracks,
    # computes per-track velocity and straightness, applies the
    # infection QC, and writes motility_plots plus a well-level summary
    # table. It only fires when timelapse is also True, and it is what
    # reveals the Motility setting categories. Default False.
    'motility_analysis': False,

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where
    # -1 means every core. Raise it to shorten CPU-bound steps until RAM
    # or disk I/O saturates. Note the measure-and-crop pipeline
    # overrides your value with cpu_count()-4. Defaults vary by
    # pipeline: cpu_count()-4, -1, or None.
    'n_jobs': 28,

    # (bool) - Percentile-normalize each image channel (2nd to 98th
    # percentile, clipped to 0-1) before display or model input; in the
    # activation-map tool this rescales the image the CAM/saliency
    # heatmap is drawn over. Turn it on when raw channels are too dim to
    # read under the overlay. Affects display and input scaling only,
    # never stored pixels. Default True.
    'normalize': True,

    # (bool) - Normalize images before plotting.
    'normalize_plots': True,

    # (float) - Cellpose cell-probability threshold for the nucleus
    # channel, passed straight to model.eval as cellprob_threshold. A
    # pixel must exceed it to join a mask, so raising it shrinks masks
    # and drops dim nuclei, while lowering it grows masks and recovers
    # faint ones along with more debris. Useful range about -6 to 6;
    # default 0.
    'nucleus_CP_prob': 0,

    # (float) - Cellpose flow_threshold for nucleus masks: the maximum
    # allowed error between a mask's recomputed flows and the network's
    # predicted flows. Lowering it discards more irregularly shaped
    # nuclei, giving fewer but cleaner objects; raising it keeps nearly
    # everything Cellpose proposes. Typical range 0 to 3; spaCR default
    # 1.0, which is permissive.
    'nucleus_FT': 1.0,

    # (float) - Multiplied by nucleus_background to set the intensity a
    # bright pixel must reach before normalization stops raising the
    # upper clip point; spaCR walks the 98th to 99.5th percentiles of
    # the non-zero nucleus channel and takes the first that meets it,
    # falling back to the 99.5th. A higher value forces a HIGHER upper
    # clip, so contrast is stretched less and bright nuclei are
    # protected from saturating; a lower value picks a lower clip point,
    # stretching dim nuclei harder but blowing out bright ones sooner.
    # Default 10.
    'nucleus_Signal_to_noise': 10,

    # (float) - Splitting threshold expressed as a multiple of the
    # median nucleus area in each field: only objects larger than this
    # multiple (and larger than nucleus_min_object_area) are candidates
    # for watershed splitting. Lower it toward 1.5 to split more
    # aggressively, raise it to break up only obvious clumps. Default
    # 2.0; used only when nucleus_intensity_split is True.
    'nucleus_area_multiplier': 2.0,

    # (int) - Raw intensity value treated as background in the nucleus
    # channel. When remove_background_nucleus is True, every pixel below
    # it is zeroed before normalization; it is also multiplied by
    # nucleus_Signal_to_noise to set the upper-clip target. Raise it for
    # images with high offset or autofluorescence, lower it if dim
    # nuclei disappear. Default 100.
    'nucleus_background': 100,

    # (int or None) - Zero-indexed raw acquisition channel segmented
    # into nucleus masks, and the channel that nucleus_background,
    # nucleus_Signal_to_noise and remove_background_nucleus apply to.
    # None means no nucleus masks, hence no nucleus table, no
    # cell-to-nucleus linking, and nothing subtracted from the cytoplasm
    # mask. Set it whenever a DNA stain was acquired. Default None.
    'nucleus_channel': None,

    # (int or None) - Expected nucleus diameter in pixels, used by
    # Cellpose 4 to rescale the image by 30/diameter before segmenting.
    # None segments at native scale. Nuclei are usually the smallest
    # object you segment, so this is the one most likely to need setting
    # on low-magnification plates. spacr.diameter.estimate_diameters
    # proposes a value. Default None.
    'nucleus_diameter': None,

    # (bool) - Merge touching nucleus labels when the mean intensity
    # along their shared boundary is at least as high as the dimmer
    # object's own intensity statistic - i.e. there is no dark seam
    # between them, so the split is spurious. Controlled by
    # nucleus_intensity_threshold_method and
    # nucleus_intensity_percentile. Default False; enable when Cellpose
    # over-segments single nuclei.
    'nucleus_intensity_merge': False,

    # (int) - Percentile of each nucleus's own pixel intensities used as
    # the merge reference when nucleus_intensity_threshold_method is
    # 'percentile'. Higher values (90) demand a very bright shared
    # boundary and merge almost nothing; lower values (50) merge
    # readily. Range 0-100, default 75. Ignored when the method is
    # 'mean'.
    'nucleus_intensity_percentile': 75,

    # (bool) - Enable watershed splitting of over-large nucleus labels:
    # objects bigger than nucleus_area_multiplier times the field's
    # median nucleus area are cut at distance-transform maxima spaced
    # nucleus_min_distance apart. Despite the name it uses shape and
    # area, not intensity. Default False; enable when clumps of touching
    # nuclei are labelled as one object.
    'nucleus_intensity_split': False,

    # (str) - Which statistic of the dimmer of two touching nuclei the
    # shared-boundary intensity is compared against when
    # nucleus_intensity_merge is on. 'mean' (default) uses that object's
    # mean intensity; 'percentile' uses its
    # nucleus_intensity_percentile-th percentile, which at the default
    # 75 is stricter and merges fewer pairs. Ignored when
    # nucleus_intensity_merge is False.
    'nucleus_intensity_threshold_method': 'mean',

    # (int or None) - Maximum nucleus area in pixels^2; after
    # segmentation any label larger than this is deleted and the
    # remaining nuclei are renumbered. Use it to drop unsplit clumps of
    # touching nuclei or blowouts that swallow much of a field. 0 (the
    # default) or None disables it; there is no cap otherwise.
    'nucleus_max_area': 0,

    # (int) - Drops the brightest nuclei per field: objects whose mean
    # nucleus-channel intensity exceeds this percentile of the per-field
    # distribution of object means are removed. Useful against saturated
    # debris and staining artefacts. Range 0-100; 100 (the default)
    # disables it, and any lower value always removes some objects.
    'nucleus_max_intensity_percentile': 100,

    # (int) - Minimum nucleus area in pixels^2, applied twice: passed to
    # Cellpose as min_size so small masks are never emitted, then
    # re-applied to the label image so any surviving object below it is
    # deleted and the rest renumbered. Raise it to drop debris and
    # fragments. 0 (default) disables both filters.
    'nucleus_min_area': 0,

    # (int) - Minimum separation in pixels between watershed seeds when
    # splitting over-large nucleus labels; seeds are local maxima of the
    # object's distance transform. Set it near the radius of one nucleus
    # - too small shatters nuclei into fragments, too large yields a
    # single seed so nothing splits. Default 10; used only when
    # nucleus_intensity_split is True.
    'nucleus_min_distance': 10,

    # (int) - Drops the dimmest nuclei per field: spaCR takes the mean
    # nucleus-channel intensity of every object surviving the area and
    # border filters and removes those below this percentile of that
    # per-field distribution. Because it is relative, any value above 0
    # always removes some objects. Range 0-100; 0 (default) disables it.
    'nucleus_min_intensity_percentile': 0,

    # (int) - Absolute floor in pixels^2 on the watershed split
    # threshold: objects at or below it are never split, even when
    # nucleus_area_multiplier times the field's median area would fall
    # lower. Raise it to protect small nuclei in fields where the median
    # object is tiny. Default 100; used only when
    # nucleus_intensity_split is True.
    'nucleus_min_object_area': 100,

    # (str) - Which weights segment nuclei. 'cpsam' or a path to your
    # own Train Cellpose checkpoint; there is no third option, because
    # Cellpose 4 removed every pre-SAM model. 'nuclei'/'nucleus' from an
    # older settings file is accepted and mapped to 'cpsam'. Set
    # nucleus_diameter rather than expecting a nucleus-specific model -
    # diameter is the parameter Cellpose 4 still acts on. Default
    # 'cpsam'.
    'nucleus_model_name': 'cpsam',

    # (float) - Merge two touching nucleus labels when their shared
    # boundary covers at least this fraction of the smaller object's
    # perimeter. Low non-zero values merge aggressively (0.1 joins
    # barely-touching nuclei); high values only fuse objects sharing
    # most of an edge. Range 0-1; 0 (default) disables perimeter
    # merging. Use it when one nucleus is split into fragments.
    'nucleus_perimeter_fraction': 0,

    # (bool) - After segmentation, delete every nucleus label touching
    # any of the four image edges, then renumber the rest. Enable it
    # when measuring nucleus area or total intensity, since clipped
    # nuclei bias those downward; leave it off for counts or positions,
    # as it discards real objects at every field boundary. Default
    # False.
    'nucleus_remove_border_objects': False,

    # (float) - Cellpose cellprob_threshold. Pixels whose predicted
    # probability of belonging to an object fall below it are excluded,
    # so raising it shrinks masks and drops faint organelles, while
    # lowering it grows masks and recovers dim ones along with more
    # false positives. Useful range roughly -6 to 6. Default 0.0.
    'organelle_CP_prob': 0.0,

    # (float) - Cellpose flow_threshold: the maximum error allowed
    # between a candidate mask's flows and the network's prediction.
    # Lowering it discards more oddly shaped masks (stricter, fewer
    # objects); raising it keeps irregular ones. Default 0.4. Raise it
    # when real organelles with non-round shapes are being thrown away.
    'organelle_FT': 0.4,

    # (int) - Side length in pixels of the local neighbourhood used to
    # compute the adaptive threshold; must be odd. Small blocks track
    # fine illumination changes but can carve holes out of large
    # organelles; large blocks behave more like a global threshold. A
    # few times the object diameter is a sensible starting point.
    # Default 51.
    'organelle_adaptive_block_size': 51,

    # (float) - Subtracted from each local mean to form the adaptive
    # threshold, so a pixel is foreground when it exceeds local_mean
    # minus this value. Raising it therefore lowers the bar and segments
    # MORE, not less; use small or negative values to be stricter. It is
    # in raw image intensity units, so a value tuned for 16-bit data
    # will flood ridge and ring modes, which threshold a 0-1 response.
    # Default 5.
    'organelle_adaptive_offset': 5,

    # (float) - Split trigger for organelle_intensity_split: only
    # objects larger than this multiple of the median organelle area in
    # the same field are candidates for watershed splitting. Drop it
    # toward 1.5 to split more aggressively, raise it to cut only
    # obvious clumps. Default 2.0. Currently inert: the organelle mask
    # writer never runs the merge/split stage.
    'organelle_area_multiplier': 2.0,

    # (int) - Zero-indexed raw acquisition channel segmented into
    # organelle masks by whichever organelle_method is chosen (otsu,
    # adaptive, log, dog, ridge, hysteresis, cellpose, unet). Setting it
    # to an integer adds an organelle mask plane to merged/ and unlocks
    # the Organelle setting categories in the GUI; None skips organelle
    # segmentation entirely. Default None.
    'organelle_channel': None,

    # (bool) - Rescale each image to 0-1 on its 0.5/99.5 percentiles,
    # then run contrast-limited adaptive histogram equalisation before
    # segmentation. Pulls dim organelles in dark corners up to the same
    # working contrast as bright ones, at the cost of amplifying
    # background noise and destroying absolute intensity comparability
    # between fields. Default False.
    'organelle_clahe': False,

    # (float) - Contrast ceiling for CLAHE, range 0-1: each tile's
    # histogram is clipped at this height before equalisation, so higher
    # values permit stronger local stretching and more noise
    # amplification. 0.01 is gentle, 0.03-0.05 is aggressive. Only read
    # when organelle_clahe is True. Default 0.01.
    'organelle_clahe_clip_limit': 0.01,

    # (float) - (DEPRECEATED) Expected organelle diameter in pixels. The
    # Cellpose-SAM path used for organelles calls model.eval with
    # diameter=None, and no classical method sizes its kernels from it,
    # so changing this value has no effect on organelle masks. Bound
    # object size with organelle_min_size / organelle_max_size instead.
    # Default 30.
    'organelle_diameter': 30,

    # (float) - Largest Gaussian scale searched by
    # Difference-of-Gaussians blob detection, in pixels. Scales are
    # stepped up from the low sigma by a factor of 1.6 until this bound,
    # so widening the gap costs more passes but covers a wider range of
    # spot sizes. Raise it to catch larger spots. Default 3.0; must
    # exceed organelle_dog_sigma_low.
    'organelle_dog_sigma_high': 3.0,

    # (float) - Smallest Gaussian scale searched by
    # Difference-of-Gaussians blob detection, in pixels; it sets the
    # lower bound on detectable spot size (radius about sigma times
    # sqrt(2)). Raise it to suppress fine noise, lower it to catch the
    # smallest puncta. Default 1.0. The detection cutoff itself comes
    # from organelle_log_threshold, not from a dog-specific key.
    'organelle_dog_sigma_low': 1.0,

    # (int) - Fill interior holes up to this area in square pixels after
    # thresholding, so a darker centre does not turn one organelle into
    # a donut. Only applied in irregular mode. Raise it when large
    # organelles come out hollow; keep it low or 0 when the hollow
    # centre is real biology. Default 64.
    'organelle_fill_holes': 64,

    # (float) - Strong threshold that seeds hysteresis segmentation -
    # only components containing a pixel above it survive at all, then
    # they grow outward down to organelle_hysteresis_low. Values below
    # 1.0 are read as a percentile of the smoothed image (0.6 = 60th
    # percentile); 1.0 or above is absolute. Raise it to keep only
    # confidently bright filaments. Default 0.6.
    'organelle_hysteresis_high': 0.6,

    # (float) - Weak threshold for hysteresis segmentation: pixels above
    # it are kept only where they connect to a seed above
    # organelle_hysteresis_high. Values below 1.0 are read as a fraction
    # and converted to that percentile of the smoothed image (0.2 = 20th
    # percentile); 1.0 or above is an absolute intensity. Lower it to
    # trace filaments further into their dim tails. Default 0.2.
    'organelle_hysteresis_low': 0.2,

    # (bool) - Merge two touching organelle labels when the mean
    # intensity along their shared boundary is at least the interior
    # reference of the dimmer object - i.e. there is no real dark edge
    # between them. Reach for it when thresholding has cut one organelle
    # into pieces. Default False. Currently inert: the organelle mask
    # writer never runs the merge/split stage.
    'organelle_intensity_merge': False,

    # (int) - Percentile (0-100) of an object's interior intensity used
    # as the merge reference when
    # organelle_intensity_threshold_method='percentile'; ignored for
    # 'mean'. Higher values raise the bar the shared boundary must
    # clear, so fewer pairs merge. Default 75. Currently inert: the
    # organelle mask writer never runs the merge/split stage.
    'organelle_intensity_percentile': 75,

    # (bool) - Split organelle labels whose area exceeds
    # max(organelle_area_multiplier times the median object area,
    # organelle_min_object_area), using a distance-transform watershed
    # seeded by local maxima. Enable when neighbouring puncta are fused
    # into single oversized labels. Default False. Currently inert: the
    # organelle mask writer never runs the merge/split stage.
    'organelle_intensity_split': False,

    # (str) - Reference statistic for organelle_intensity_merge: 'mean'
    # compares the shared-boundary intensity to the mean interior
    # intensity of the dimmer object; 'percentile' compares it to that
    # object's organelle_intensity_percentile value instead. A high
    # percentile makes merging much stricter. Default 'mean'. Currently
    # inert: the organelle mask writer never runs the merge/split stage.
    'organelle_intensity_threshold_method': 'mean',

    # (float) - Largest Gaussian scale searched by LoG blob detection,
    # in pixels; blob radius is about sigma times sqrt(2), so 10 caps
    # detection near a 14 px radius. Raise it to catch large puncta, at
    # a runtime cost since the filter is evaluated once per scale.
    # Default 10; must exceed organelle_log_min_sigma.
    'organelle_log_max_sigma': 10,

    # (float) - Smallest Gaussian scale searched by LoG blob detection,
    # in pixels; the detected blob radius is about sigma times sqrt(2),
    # so sigma 1 finds roughly 1.4 px radius puncta. Raise it to ignore
    # single-pixel noise, lower it to catch the smallest spots. Default
    # 1; must stay below organelle_log_max_sigma.
    'organelle_log_min_sigma': 1,

    # (int) - How many Gaussian scales are evaluated between
    # organelle_log_min_sigma and organelle_log_max_sigma. More scales
    # resolve a wider spread of spot sizes, but the filter runs once per
    # scale so runtime grows linearly. Default 10; drop to 3-5 when spot
    # size is uniform and you need speed.
    'organelle_log_num_sigma': 10,

    # (float) - Minimum LoG/DoG response a local maximum must reach to
    # count as a blob, measured after the image is percentile-normalised
    # to 0-1, so it behaves like a contrast fraction. Lower it to pick
    # up fainter puncta along with more noise; raise it to keep only
    # bright ones. Default 0.01. The 'dog' method reads this key too.
    'organelle_log_threshold': 0.01,

    # (bool) - Zero every pixel outside the cell mask before segmenting,
    # so organelles can only be found inside cells and extracellular
    # debris cannot generate objects. Needs cell_mask_stack/ to already
    # exist alongside the organelle source; if it is missing spacr
    # prints a warning and carries on unmasked rather than failing.
    # Default False.
    'organelle_mask_within_cells': False,

    # (int or None) - Post-segmentation area ceiling in square pixels;
    # larger objects are deleted. Use it to reject fused clumps and
    # saturated debris. Default 0, and either 0 or None disables it.
    # Note this is the shared object filter (used by the Qt live
    # preview); the batch organelle mask writer caps size with
    # organelle_max_size instead.
    'organelle_max_area': 0,

    # (int or None) - Drops organelle objects whose mean intensity
    # exceeds this percentile of all organelle mean intensities in the
    # same field, so it removes roughly the brightest (100 minus value)
    # percent. Range 0-100; 100 or None disables it (None is read as the
    # default 100, it does not error). Applied by the shared Qt
    # live-preview filter - the batch organelle mask pipeline does not
    # run this filter. Default 100. Use it to reject saturated dust and
    # imaging artefacts.
    'organelle_max_intensity_percentile': 100,

    # (int or None) - Upper area bound in square pixels applied to the
    # finished label image; any object above it is deleted outright, not
    # split. Use it to drop fused clumps, saturated debris and
    # background regions that Otsu swallowed into one blob. Set it below
    # your largest genuine organelle and you will silently lose real
    # objects. Default None (no limit).
    'organelle_max_size': None,

    # (str) - Segmentation backend, validated against
    # organelle_morphology: 'otsu' (one global threshold), 'adaptive'
    # (local threshold), 'log'/'dog' (blob detection), 'ridge' (tubeness
    # filter, network only), 'hysteresis' (dual threshold, network
    # only), 'cellpose' (pretrained model), 'unet' (your own model,
    # network only). Classical methods run on CPU across n_jobs workers;
    # cellpose and unet run on the GPU. Default 'otsu'.
    'organelle_method': 'otsu',

    # (int) - Post-segmentation area floor in square pixels; smaller
    # objects are deleted and the mask relabelled. Raise it to clear
    # noise specks left by thresholding. Default 0 (disabled). Note this
    # is the shared object filter (used by the Qt live preview); the
    # batch organelle mask writer does its own size filtering with
    # organelle_min_size.
    'organelle_min_area': 0,

    # (int) - Minimum separation in pixels between watershed seeds when
    # splitting oversized organelles; distance-transform peaks closer
    # than this collapse into a single seed. Raise it to stop one
    # organelle being shredded into fragments, lower it to separate
    # tightly packed puncta. Default 10. Currently inert: the organelle
    # mask writer never runs the merge/split stage.
    'organelle_min_distance': 10,

    # (int) - Drops organelles whose mean intensity falls below this
    # percentile of all organelle mean intensities in the same field. It
    # is relative, not absolute, so it always removes roughly this share
    # of the dimmest objects even in a clean image. Range 0-100, 0
    # disables. Default 0. Use it to cull background-level detections.
    'organelle_min_intensity_percentile': 0,

    # (int) - Absolute area floor in square pixels for the split step:
    # an object is only split if its area also clears this, so small
    # objects survive even when the median-based threshold is tiny.
    # Raise it to protect genuine small organelles from being cut in
    # half. Default 100. Currently inert: the organelle mask writer
    # never runs the merge/split stage.
    'organelle_min_object_area': 100,

    # (int) - (Depreceated) Minimum object area in square pixels. Most
    # classical segmenters and the U-Net discard smaller components
    # during segmentation via remove_small_objects (the LoG/DoG spot
    # methods do not, and the ring method applies a quarter of it, floor
    # 3, to its edge image), and the value is always applied again to
    # the finished label image. Despite the marker it is still live -
    # raise it to clear dim specks and hot pixels, lower it to keep
    # faint puncta. Default 10; 0 disables.
    'organelle_min_size': 10,

    # (str) - Cellpose model used when organelle_method='cellpose'.
    # Cellpose 4 provides only 'cpsam'; the pre-SAM names are accepted
    # and mapped to it. Change this only to point at a custom
    # CPSAM-architecture checkpoint. Default 'cpsam'.
    'organelle_model_name': 'cpsam',

    # (int) - Radius in pixels of the disk used for morphological
    # cleanup. In irregular mode it also sets the pre-smoothing sigma
    # (radius/2) and drives a closing then an opening, bridging gaps and
    # erasing protrusions thinner than the disk; network modes use half
    # this radius for closing only. Raise it to smooth ragged outlines,
    # lower it to preserve fine detail. Default 3.
    'organelle_morph_radius': 3,

    # (str) - Shape family of the target organelle; picks the
    # segmentation pipeline and restricts which organelle_method values
    # are legal. 'spots' = punctate (vesicles, lipid droplets),
    # 'network' = filamentous (mitochondria, ER tubules), 'irregular' =
    # solid blobby (Golgi, lysosomes), 'ring' = hollow (endosomes,
    # autophagosomes). Default 'spots'. An unsupported morphology/method
    # pair raises before any image is loaded.
    'organelle_morphology': 'spots',

    # (str) - How the ridge-filter response is binarised: 'otsu' takes
    # one global cut-off from the response histogram, 'adaptive' uses a
    # local threshold (organelle_adaptive_block_size / _offset) and
    # keeps faint filaments in dim regions at the cost of extra
    # background. Only read by organelle_method='ridge'; anything
    # unrecognised silently falls back to otsu. Default 'otsu'.
    'organelle_network_threshold': 'otsu',

    # (float) - Merge two touching organelle labels when their shared
    # boundary is at least this fraction of the smaller object's
    # perimeter. Range 0-1; push it toward 1 to merge only
    # near-fully-fused pairs, lower it to aggressively glue neighbours.
    # Default 0 (disabled). Currently inert: the organelle mask writer
    # never runs the merge/split stage.
    'organelle_perimeter_fraction': 0,

    # (bool) - Delete every organelle label touching any of the four
    # image edges, in the final post-processing step before counting and
    # saving, so partly imaged objects do not bias area and intensity
    # statistics. Costs you real objects around the FOV rim, which
    # matters more the larger the organelle. Default False.
    'organelle_remove_border': False,

    # (bool) - Delete organelle labels touching any image edge during
    # the shared post-segmentation filter (the Qt live preview path).
    # The batch organelle mask writer does the same job from
    # organelle_remove_border, so set that one for a real run. Default
    # False. Enable to keep clipped rim objects out of area and
    # intensity statistics.
    'organelle_remove_border_objects': False,

    # (bool) - (DEPRECEATED) Passed to Cellpose as resample: when True
    # the flows are recomputed at full resolution instead of on the
    # downsampled grid, giving smoother and slightly more accurate
    # outlines for a little extra time. Still forwarded to model.eval on
    # the organelle path. Default True; leave it alone unless you are
    # chasing speed.
    'organelle_resample': True,

    # (str) - Which vesselness filter enhances filaments before
    # thresholding: 'frangi' (classic, crisp on well-separated tubules),
    # 'sato' (more tolerant of varying thickness), 'meijering' (tuned
    # for thin neurite-like fibres). All run with black_ridges=False,
    # i.e. bright filaments on a dark background. Default 'frangi'; try
    # 'sato' when frangi drops faint filaments.
    'organelle_ridge_filter': 'frangi',

    # (list of float) - Scales in pixels at which the vesselness filter
    # looks for tubular structures; each value should sit near the
    # half-width of a filament and the responses are combined across
    # scales. Add larger values to pick up thick bundles, keep them
    # small for fine tubules. Default [1, 2, 3]; longer lists cost
    # proportionally more time.
    'organelle_ridge_sigmas': [1, 2, 3],

    # (str) - How detected ring walls become solid objects: 'flood'
    # fills every background component that does not touch the image
    # border - accurate, but leaks through any gap in the wall - while
    # 'convex' takes the convex hull of each wall component, which
    # tolerates broken rings but overshoots concave shapes. Default
    # 'flood'; switch to 'convex' when rings come out unfilled.
    'organelle_ring_fill_method': 'flood',

    # (float) - Shape gate for ring mode: for each filled object spacr
    # computes abs(mean wall intensity minus mean lumen intensity)
    # divided by the object's mean intensity, and deletes anything below
    # this value. Raise it to keep only clearly hollow objects, lower it
    # to also accept partly filled ones. 0 disables the gate. Default
    # 0.1.
    'organelle_ring_min_prominence': 0.1,

    # (float) - Low sigma of the Difference-of-Gaussians band-pass that
    # highlights ring walls, in pixels; set it near the wall thickness
    # so the wall survives the high-pass. Too small and pixel noise is
    # retained, too large and the wall blurs into the lumen and the ring
    # stops being detected as hollow. Default 1.0; must be below
    # organelle_ring_sigma_outer.
    'organelle_ring_sigma_inner': 1.0,

    # (float) - High sigma of the ring Difference-of-Gaussians
    # band-pass, in pixels; it sets the coarse scale that gets
    # subtracted, so keep it around the ring's outer radius. Widen the
    # gap from organelle_ring_sigma_inner to enhance larger rings,
    # narrow it for tight vesicles. Default 3.0; must exceed
    # organelle_ring_sigma_inner.
    'organelle_ring_sigma_outer': 3.0,

    # (bool) - Roll a ball of organelle_rolling_ball_radius under the
    # intensity surface, subtract the resulting background estimate and
    # clip negatives to zero, before any segmentation runs. Flattens
    # uneven illumination and haze so a single global threshold works
    # across the whole FOV. Costs real time per image. Default False.
    'organelle_rolling_ball': False,

    # (int) - Radius in pixels of the rolling ball background estimator.
    # It must be clearly larger than the biggest real organelle or the
    # ball follows the objects and subtracts them away; too large and it
    # stops tracking the illumination gradient. A few times the object
    # diameter is a good start. Default 50; runtime grows steeply with
    # radius.
    'organelle_rolling_ball_radius': 50,

    # (bool) - Reduce each thresholded network to a one-pixel-wide
    # skeleton (dilated by 1 px so it stays connected) and label that
    # instead of the filled filaments. Measured areas then track network
    # length rather than filament thickness. Enable for topology and
    # length analysis, disable to measure filament mass. Default False.
    'organelle_skeletonize': False,

    # (int) - Radius in pixels of the disk used for white top-hat
    # filtering before otsu/adaptive spot thresholding; it removes
    # everything broader than the disk, flattening haze and background.
    # Set it just above the largest genuine spot - too small erases the
    # spots themselves, too large leaves background in. Default 5.
    # Ignored by the log and dog methods.
    'organelle_tophat_radius': 5,

    # (str or None) - Path to a serialised PyTorch model used when
    # organelle_method='unet'. It must be a torch.load-able whole
    # module, not a state_dict, and take z-scored (B,1,H,W) input
    # returning (B,1,H,W) logits; extra output channels are silently
    # ignored except the first. A missing or invalid path raises before
    # segmentation starts. Default None.
    'organelle_unet_model_path': None,

    # (float) - Probability cut-off applied to the U-Net's sigmoid
    # output, range 0-1. Lower it to grow the predicted network and
    # recover faint branches at the cost of false positives; raise it to
    # keep only confident pixels, which tends to break weak connections.
    # Objects below organelle_min_size are still removed afterwards.
    # Default 0.5.
    'organelle_unet_threshold': 0.5,

    # (bool) - Split touching spots instead of labelling each connected
    # blob once. Under otsu/adaptive it runs a distance-transform
    # watershed with seeds at least 5 px apart; under log/dog it grows a
    # watershed from each blob centre instead of stamping a disk whose
    # radius comes from that blob's own sigma (round(sigma*sqrt(2)),
    # minimum 1 px). Turn it off when single spots are being fragmented.
    # Default True.
    'organelle_watershed_spots': True,

    # (float) - Cellpose cellprob_threshold for the pathogen channel: a
    # pixel is claimed by a mask only if its predicted object
    # probability exceeds this. Lower it (toward -6) to recover dim or
    # small parasites and grow mask boundaries; raise it (toward 6) to
    # shrink masks and drop faint objects. Useful range about -6 to 6.
    # Default 0.
    'pathogen_CP_prob': 0,

    # (float) - Cellpose flow_threshold for pathogen masks: a candidate
    # mask is discarded when its recomputed flows disagree with the
    # network prediction by more than this. Raise it to keep more,
    # sometimes misshapen, parasites; lower it toward 0.4 (Cellpose's
    # own default) to keep only clean, well-formed objects. Typical
    # range 0.0-3.0. Default 1.0.
    'pathogen_FT': 1.0,

    # (int) - Expected foreground-to-background ratio of the pathogen
    # channel. Multiplied by pathogen_background it gives the intensity
    # the normalisation ceiling must clear: spaCR walks percentiles 98
    # to 99.5 and takes the first that reaches it, falling back to 99.5.
    # Raise it for a higher, dimmer, less clipped ceiling; lower it for
    # more contrast. Default 10.
    'pathogen_Signal_to_noise': 10,

    # (float) - Splitting trigger expressed as multiples of the MEDIAN
    # pathogen area in the field: only labels larger than this multiple
    # (and larger than pathogen_min_object_area) go to the watershed.
    # Lower it toward 1.5 to split more aggressively, raise it to break
    # up only obvious clumps. Used only when pathogen_intensity_split is
    # True. Default 2.0.
    'pathogen_area_multiplier': 2.0,

    # (int) - Assumed background intensity of the pathogen channel in
    # raw image units. It has two jobs: when remove_background_pathogen
    # is True every pixel below it is zeroed, and it is multiplied by
    # pathogen_Signal_to_noise to set the brightness the normalisation
    # ceiling must reach. Raise it if dim haze is being segmented; lower
    # it if faint parasites vanish. Default 100.
    'pathogen_background': 100,

    # (int or None) - Zero-indexed raw acquisition channel segmented
    # into pathogen masks (Toxoplasma etc.), and the channel
    # pathogen_background, pathogen_Signal_to_noise and
    # remove_background_pathogen apply to. None disables pathogen
    # segmentation, the pathogen table, the infected-only filter
    # (uninfected) and the adjust_cells step, which needs cell, nucleus
    # and pathogen masks together. Default None.
    'pathogen_channel': None,

    # (int or None) - Expected pathogen diameter in pixels, used by
    # Cellpose 4 to rescale the image by 30/diameter before segmenting.
    # None segments at native scale. Intracellular parasites are often
    # only a few pixels across at low magnification, where rescaling
    # matters most. spacr.diameter.estimate_diameters proposes a value.
    # Default None.
    'pathogen_diameter': None,

    # (bool) - Merge two touching pathogen labels when the mean
    # intensity along their shared border is at least as bright as the
    # interior of the dimmer of the pair, i.e. there is no real
    # intensity valley between them. Enable it to repair vacuoles
    # Cellpose split down the middle. Needs an intensity image; tuned by
    # pathogen_intensity_threshold_method. Default False.
    'pathogen_intensity_merge': False,

    # (int) - Percentile, 0-100, of a pathogen's interior intensity used
    # as the merge reference when pathogen_intensity_threshold_method is
    # 'percentile'. Two touching labels merge only if their shared
    # border is at least this bright inside the dimmer object, so
    # raising it demands a brighter border and merges fewer pairs.
    # Ignored when the method is 'mean'. Default 75.
    'pathogen_intensity_percentile': 75,

    # (bool) - Enable watershed splitting of oversized pathogen labels.
    # Despite the name the split is purely geometric: objects larger
    # than max(pathogen_area_multiplier x median area,
    # pathogen_min_object_area) are cut at local maxima of their
    # distance transform. Turn it on when several parasites in one
    # vacuole are fused into a single mask. Default False.
    'pathogen_intensity_split': False,

    # (str) - How the reference brightness is computed when
    # pathogen_intensity_merge decides whether two touching labels have
    # a real edge. 'mean' compares the shared-border intensity to the
    # mean interior intensity of the dimmer object; 'percentile'
    # compares it to pathogen_intensity_percentile of that object
    # instead, which is stricter and merges fewer pairs. Default 'mean'.
    'pathogen_intensity_threshold_method': 'mean',

    # (int or None) - Maximum pathogen area in pixels squared; labels
    # larger than this are deleted after segmentation. 0, the default,
    # or None disables the filter. Use it to drop fused clumps and
    # segmentation blowouts that would otherwise dominate per-object
    # statistics - but prefer pathogen_intensity_split if you want
    # clumps separated rather than discarded.
    'pathogen_max_area': 0,

    # (int or None) - Upper end of the same per-field percentile filter:
    # pathogens whose mean intensity exceeds this percentile of the
    # field's pathogen mean intensities are deleted. 100, the default,
    # disables it. Lower it to strip saturated debris and bright
    # artefacts. Any value below 100 forces the intensity image to be
    # loaded during filtering.
    'pathogen_max_intensity_percentile': 100,

    # (int) - Minimum pathogen area in pixels squared. Passed to
    # Cellpose as min_size so undersized masks never leave segmentation,
    # then re-applied in the merge/split/filter pass. 0, the default,
    # disables it. Raise it to clear speckle and debris; set it too high
    # and small or newly divided parasites disappear.
    'pathogen_min_area': 0,

    # (int) - Minimum separation in pixels between watershed seeds when
    # splitting oversized pathogen labels; seeds are local maxima of the
    # distance transform. Raise it for fewer, larger fragments (or none,
    # leaving the object intact); lower it to cut clumps into more
    # pieces. Used only when pathogen_intensity_split is True. Default
    # 10.
    'pathogen_min_distance': 10,

    # (int) - Relative brightness cutoff, 0-100: within each field the
    # mean intensity of every surviving pathogen is ranked, and objects
    # below this percentile of that distribution are deleted. It is not
    # an absolute intensity, so how many objects go depends on the
    # object count. 0, the default, disables it. Raise it to drop dim
    # false positives.
    'pathogen_min_intensity_percentile': 0,

    # (int) - Absolute floor in pixels squared below which a pathogen
    # label is never split, whatever the median area says: the effective
    # split threshold is max(pathogen_area_multiplier x median area,
    # this value). Raise it to protect small parasites from
    # fragmentation in sparse fields. Used only when
    # pathogen_intensity_split is True. Default 100.
    'pathogen_min_object_area': 100,

    # (str or None) - Path to a custom Cellpose checkpoint used to
    # detect pathogen objects, overriding pathogen_model_name when set.
    # It must be a CPSAM-architecture checkpoint (one your own Train
    # Cellpose run produced); a Cellpose-3 CPnet file cannot load into
    # Cellpose 4. A path that does not exist stops the run rather than
    # falling back to the stock weights silently. Default None.
    'pathogen_model': None,

    # (str) - Which weights segment pathogens. 'cpsam' or a path to your
    # own Train Cellpose checkpoint. The bundled toxo_pv_lumen /
    # toxo_cyto checkpoints were Cellpose-3 CPnet and cannot load into
    # CPSAM's transformer, so they are mapped to 'cpsam' and reported.
    # The older 'pathogen_model' key still overrides this one when set.
    # Default 'cpsam'.
    'pathogen_model_name': 'cpsam',

    # (float) - Fraction, 0-1, of the SMALLER label's perimeter that two
    # touching pathogen objects must share before they are fused into
    # one. 0, the default, disables perimeter merging; values near 0.1
    # fuse almost anything that touches, while 0.5-0.8 fuse only objects
    # with a long common border. Use it to repair vacuoles Cellpose cut
    # in two.
    'pathogen_perimeter_fraction': 0,

    # (bool) - Delete any pathogen label touching the first or last row
    # or column of the image. Enable it so partially imaged parasites do
    # not enter area and intensity statistics with truncated values;
    # leave it off when parasites are sparse and losing edge objects
    # costs too much data. Default False.
    'pathogen_remove_border_objects': False,

    # (str) - Which mask pipeline runs. 'v1' is the disk-based chain
    # (rename, per-channel folders, npy, npz, mask npy, merged/) that
    # measure, annotate and every downstream tool expect, and is the
    # fully tested path. 'v2' streams from the originals and writes one
    # npy per field with masks appended in place, using roughly 60-80%
    # less disk but producing no .npz. Default 'v1'.
    'pipeline_style': 'v1',

    # (bool) - Render and save QC figures while the pipeline runs:
    # channel montages and Cellpose mask overlays during segmentation,
    # before/after filtration views and crop grids during measurement.
    # It adds figures per batch, so a full plate becomes much slower and
    # more memory-hungry; keep it for small or test_mode runs, which
    # force it on. Default False.
    'plot': False,

    # (bool) - Run the image-preparation stage before segmentation:
    # group raw files into per-field channel stacks, optionally subtract
    # background, and percentile-normalize each channel into float
    # arrays. Leave True on a fresh run; set False only when those
    # normalized arrays already exist, otherwise segmentation has
    # nothing to read. Default True.
    'preprocess': True,

    # (bool) - Shuffle the order of the per-field arrays before they are
    # grouped into normalization batches, so each batch spans plates and
    # wells instead of one acquisition block - this matters because
    # normalization percentiles are computed per batch. Forced to False
    # for timelapse runs to keep frames in sequence. Default True.
    'randomize': True,

    # (bool) - Before normalisation, zero every pixel in the cell
    # channel below cell_background. This flattens haze so the
    # percentile stretch is driven by real signal, but it also erases
    # genuinely dim cell edges and can shrink masks. Enable only once
    # cell_background is set from an actual empty region. Default False.
    'remove_background_cell': False,

    # (bool) - Before normalizing the nucleus channel, zero every pixel
    # below nucleus_background and exclude those pixels from the
    # percentile calculation. Enabling it raises contrast on real nuclei
    # and suppresses haze, but clips genuinely dim nuclei to zero so
    # they may become unsegmentable. Default False; check
    # nucleus_background against raw images first.
    'remove_background_nucleus': False,

    # (bool) - Before normalising the pathogen channel, hard-zero every
    # pixel whose raw intensity is below pathogen_background. Enable it
    # when diffuse autofluorescence inflates the low percentile and
    # Cellpose starts segmenting haze; leave it off for dim parasites,
    # since the clipping erases real signal and biases downstream
    # intensity measurements. Default False.
    'remove_background_pathogen': False,

    # (bool) - Continue an interrupted run at its last verified safe
    # boundary instead of starting over. Mask validates existing
    # per-field mask and merged arrays; Measure accepts only fields
    # complete in every owned measurements table and transactionally
    # clears partial rows before retrying; Format Converter reopens
    # every TIFF in a checkpointed field; UMAP reloads completed trial
    # scores and embedding artifacts, including an incomplete adaptive
    # round; Batch preserves settled plate/jobs and automatically
    # enables field resume for an interrupted Mask, Measure, or
    # conversion job. A checkpoint whose inputs or material settings
    # differ is refused rather than mixing incompatible output. Default
    # False.
    'resume': False,

    # (bool or list of bool) - Whether to save masks to disk. Can be a
    # list of three booleans for [cell, nucleus, pathogen]
    # independently.
    'save': True,

    # (bool) - After each batch is MIP-projected and merged into stack/,
    # either move the raw input images into src/orig/ (True) or delete
    # them so the pixels live only in stack/ (False). Set False on large
    # screens where the duplicate raw copy will not fit on disk; the
    # deletion is not reversible. Default True.
    'save_original_images': True,

    # (str) - Segmentation quality control, scored on the masks the
    # moment they are written and long before measure_crop spends hours
    # on them. 'off' skips it entirely; 'report' (the default) scores
    # every field, writes qc/segmentation_qc_OBJECT.csv under the plate
    # folder and prints a card naming the fields that are wrong and why;
    # 'flag' does the same and additionally saves a per-field flags JSON
    # for a downstream step to consume. No mode ever deletes, filters or
    # skips a field: it tells you which ones are bad and leaves the
    # decision to you. Default 'report'.
    'seg_qc': 'report',

    # (float) - Fraction of a field's objects allowed to touch the image
    # edge before the field is flagged. Objects on the edge are
    # truncated, so the crops handed to Measure are cut off and their
    # areas understate the truth. Geometry alone puts roughly two object
    # diameters' worth on the border, about 8% for 60 px cells in a 1400
    # px field, so this default is well clear of what a healthy field
    # produces. Default 0.3.
    'seg_qc_border_fraction': 0.3,

    # (float) - How far a field's object count may drift from the plate
    # median before the field is flagged, expressed as a fraction: 0.25
    # flags anything below a quarter of the median and, through its
    # reciprocal, anything above four times it. Seeding density across a
    # plate varies with a coefficient of variation of 10-30% and edge
    # wells rarely fall below half, so a four-fold departure means lost
    # focus, an empty well or a collapsed mask rather than biology.
    # Default 0.25.
    'seg_qc_count_ratio': 0.25,

    # (float) - Foreground coverage at or above which a field is called
    # confluent. That is the first half of the fusion test and the only
    # condition under which the expensive distance-transform cross-check
    # runs at all, so raising it makes QC faster and blinder while
    # lowering it costs time on sparse fields. It matches the
    # fused_fraction the diameter estimator uses, so both modules agree
    # on what a dense field is. Default 0.35.
    'seg_qc_foreground_fraction': 0.35,

    # (float) - Share of the entire field a single label may cover
    # before that label is read as evidence of fusion rather than as an
    # object. A quarter of a field is not a cell, it is a monolayer that
    # was welded into one mask, and the diameter estimator discards such
    # components for exactly the same reason. Lower it for small objects
    # on large fields; raise it only when one huge object per field is
    # real. Default 0.25.
    'seg_qc_max_object_fraction': 0.25,

    # (float) - Equivalent diameter in pixels below which an object is
    # treated as a fragment rather than a real one; it drives the
    # over-segmentation check and sets the seed floor of the fusion
    # cross-check. Lower it to two or three for punctate organelles,
    # where five-pixel objects are the actual signal rather than debris,
    # and raise it for large cells where anything that small is
    # certainly a shard. Default 5.
    'seg_qc_min_diameter': 5.0,

    # (int) - Fields holding fewer objects than this are called
    # near-empty, and their robust per-field size statistics are
    # suppressed, because a median absolute deviation taken over a
    # handful of objects is one object's opinion rather than a
    # distribution. Raise it for confluent cell plates where every field
    # should hold hundreds; drop it to 3-5 for low-multiplicity pathogen
    # channels where two objects per field is genuinely the assay.
    # Default 10.
    'seg_qc_min_objects': 10,

    # (float) - Share of a field's objects that must fall outside the
    # robust size range before the field is reported as holding two
    # populations. A single distribution's tail cannot put fifteen
    # percent of its objects five robust deviations out; only debris,
    # fused pairs or fragments can, and those are what this check is
    # looking for. Lower it if you want the card to be chattier about
    # mixed fields. Default 0.15.
    'seg_qc_outlier_fraction': 0.15,

    # (float) - How many robust standard deviations, one of which is
    # 1.4826 times the median absolute deviation, an object's diameter
    # may sit from the field median before it counts as a size outlier.
    # Median and MAD are used rather than mean and standard deviation
    # because a few pieces of debris inflate a standard deviation until
    # nothing looks unusual any more. Five is deliberately loose: real
    # size distributions are heavier tailed than Gaussian, so three
    # would flag part of every healthy field. Default 5.
    'seg_qc_outlier_mad': 5.0,

    # (float) - Fraction of failing fields at which the scorecard's
    # verdict for the whole plate flips from warn to fail. Ten percent
    # is roughly one column of a 96-well plate: below that you can drop
    # the bad fields and still have the experiment, and above it what
    # Measure would produce is no longer the experiment you ran. It
    # changes the printed verdict only, never which fields are
    # processed. Default 0.1.
    'seg_qc_plate_fail_fraction': 0.1,

    # (float) - Fold change in a field's median object diameter,
    # measured against the plate median, that marks it as fused or
    # shattered when its object count has moved the opposite way. The
    # default is the square root of two on purpose: two objects welded
    # into one have exactly 1.41 times the equivalent diameter of one,
    # and one object split in two has the reciprocal, so this is the
    # signature itself rather than an arbitrary tolerance. Default 1.4.
    'seg_qc_size_ratio': 1.4,

    # (float) - How many objects the distance transform has to resolve
    # per mask object, in a field already judged confluent, before those
    # masks are called fused. Two means every mask object contains at
    # least two inscribed-circle maxima on average, which is the
    # smallest fusion worth catching: neighbouring cells merged in
    # pairs. Raise it toward five to report only wholesale collapse of a
    # monolayer into slabs. Default 2.
    'seg_qc_split_ratio': 2.0,

    # (float) - Share of a field's objects that may be smaller than
    # seg_qc_min_diameter before the whole field is called
    # over-segmented. Cellpose shattering one cell into a dozen shards
    # takes this close to one, while a healthy field carrying a little
    # debris sits well below a third, which is where the default sits.
    # Default 0.3.
    'seg_qc_tiny_fraction': 0.3,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (float) - Minimum overlap, as an intersection-over-union between 0
    # and 1, for a label in one plane to be treated as the same object
    # as a label in the plane below when z_segmentation_mode is
    # 'stitch'. Raising it splits objects that drift or change shape
    # between planes into several shorter ones; lowering it fuses
    # neighbouring objects that merely overlap in projection. Matching
    # is one-to-one, so when two objects both overlap the same object
    # below only the better match inherits its label and the other
    # starts a new one. Ignored by the other two modes. Default 0.25.
    'stitch_threshold': 0.25,

    # (bool or None) - What happens when a step hits a problem it could
    # technically survive. None (the default) defers to the
    # SPACR_STRICT_ERRORS environment variable, which is how a cluster
    # turns this on for a whole batch without editing every settings
    # file; False and True are explicit per-run choices and override it.
    # When off, the failure is recorded in the run ledger, printed in
    # the end-of-run summary and stamped into the artifact's run_status,
    # and the run continues on the items that worked. When on, a
    # swallowed setup or configuration error - an unreadable path, a
    # missing column, a database that will not open - raises
    # immediately, so a batch job stops at the first sign its inputs are
    # wrong instead of producing a plausible-looking partial result.
    # Per-item failures such as one corrupt image are still survived
    # either way. Default None.
    'strict_errors': None,

    # (list or None) - Parent compartments to roll organelle
    # measurements up into. Accepts 'cell', 'nucleus', 'pathogen',
    # 'cytoplasm', each writing a <parent>_organelle_summary table of
    # per-parent organelle count, total/mean/std area, area fraction of
    # the parent, mean/std eccentricity and solidity, axis lengths and
    # per-channel mean intensity; include 'organelle' to also save the
    # raw per-organelle table. Default 'cell'; None writes nothing.
    'summarize_organelles_by': 'cell',

    # (int or None) - Index of the time axis in the incoming array, as
    # an alternative to spelling out the whole order in t_axis_order;
    # the z axis is then taken to be the other of the two leading axes,
    # or whatever z_axis says. Use it for an acquisition whose axes are
    # not in either of the two standard orders. When both this and
    # t_axis_order are set they must agree, and spaCR stops if they do
    # not rather than silently preferring one. Default None.
    't_axis': None,

    # (str or None) - Which of the two leading axes of your data is time
    # and which is z: 'TZYX' for a stack per timepoint, 'ZTYX' for a
    # time series per plane. Both are written by real microscopes and
    # the array shape cannot distinguish them, so spaCR refuses to guess
    # and stops until you say which you have. Getting this wrong does
    # not crash: it links each object to whatever sits above it in the
    # next z plane and reports that as motion, which produces smooth,
    # entirely plausible, entirely fictional trajectories. A flat 2-D
    # time series with no z at all is spelled 'TYX' - the one value that
    # means "there is no z axis", since every other spelling claims one,
    # so a flat acquisition is never the result of an ambiguity resolved
    # in its favour. Leave it None only when you are setting t_axis
    # directly instead. Default None.
    't_axis_order': None,

    # (float) - Minimum overlap, as an intersection-over-union between 0
    # and 1, for an object at one timepoint to be treated as the same
    # object at the next when t_track_backend is 'iou'. Raising it
    # breaks a moving or growing object into several short tracks;
    # lowering it fuses neighbouring objects whose volumes happen to
    # touch. Kept separate from stitch_threshold on purpose, because
    # consecutive z planes and consecutive timepoints do not overlap by
    # anything like the same amount. Matching is one-to-one, so two
    # objects cannot both inherit one identity. Default 0.25.
    't_link_threshold': 0.25,

    # (float or None) - How far an object may move between consecutive
    # timepoints and still be considered the same object, in image
    # pixels, for the distance-based backends. The z component of every
    # displacement is multiplied by the anisotropy first, so that a
    # one-plane move on a stack with a 5x z step counts as five pixels
    # rather than one, which is what stops objects five times too far
    # apart from linking. Set this or t_max_displacement_um but not
    # both; spaCR will not pick a default, because too large fuses
    # neighbours into one track and too small breaks one object into a
    # track per frame. Default None.
    't_max_displacement_px': None,

    # (float or None) - The same maximum between-frame movement as
    # t_max_displacement_px but expressed in micrometres, which is
    # usually the number you actually know from the biology. It needs
    # voxel_size_z_um and voxel_size_xy_um to convert, and once those
    # are set it is the safer of the two because the anisotropy is
    # already baked into the physical coordinates instead of being
    # applied as a correction. Set this or t_max_displacement_px but
    # never both, since they are one gate in two units. Default None.
    't_max_displacement_um': None,

    # (bool) - Collapse each timepoint's z-stack to one plane before
    # linking, so tracking happens on the projection while segmentation
    # still happened on the volume. Turn it on when the volumetric
    # linking is too slow or you do not trust the anisotropy, and accept
    # the cost: two objects that sit above one another become one object
    # and nothing computed downstream can tell that it happened. It does
    # not enable the backends spaCR cannot drive on volumes, which are
    # refused whatever this is set to. Default False.
    't_project_for_tracking': False,

    # (bool) - Treat each field as a time series of z-stacks rather than
    # as one flat image, turning on the axis-order, backend and
    # displacement controls below. Off, spaCR runs whatever 2-D or 3-D
    # path you already had and not one line of the 4-D code executes, so
    # your masks and tracks are identical to a run from before this
    # setting existed. On, spaCR requires an array that still has both a
    # time axis and a z axis when it reaches segmentation and stops with
    # an error naming the cause if it does not; the standard image
    # ingest writes one maximum-intensity plane per timepoint while
    # organising the raw files, so this only has something to work with
    # when you hand spaCR whole volumes through the Python API. Default
    # False.
    't_stack': False,

    # (str) - Which linker joins objects between consecutive timepoints
    # once they have been segmented in 3-D. 'iou' overlaps whole volumes
    # and needs no distance, no anisotropy and no tuning, but loses
    # anything that moves further than its own width between frames;
    # 'centroid' links nearest centroids under the displacement gate
    # below and handles fast movement, at the cost of needing that gate
    # set correctly; 'trackpy' does the same through trackpy's linker.
    # The btrack, trackastra and ultrack backends all handle volumes
    # upstream but spaCR's adapters for them require a flat time series,
    # so asking for one of them on volumetric masks stops the run
    # instead of quietly flattening it. Default 'iou'.
    't_track_backend': 'iou',

    # (int) - How many plate/well/field image sets are copied into a
    # test/ folder when test_mode is on; every channel file belonging to
    # a chosen set is copied together. Raise it for a broader smoke
    # test, lower it for a faster one. Forced to 1 for timelapse runs so
    # a full sequence stays intact. Default 10.
    'test_images': 10,

    # (bool) - Run the pipeline on a small random subset instead of the
    # whole folder. Mask generation copies test_images (default 10)
    # complete image sets into <src>/test and works there; measure_crop
    # copies test_nr (default 10) merged arrays into test/merged. Both
    # also force verbose and plot on. Use it to check channel
    # assignment, diameters and thresholds before committing to a full
    # plate. Default False.
    'test_mode': False,

    # (bool) - Treat each well/field as a time series instead of
    # independent images: files are grouped into time stacks,
    # randomization is switched off, per-channel movies are written,
    # objects in timelapse_objects are tracked across frames, a timeID
    # column is added to the measurement tables, and measure_crop stops
    # writing single-object PNGs. Only enable when filenames carry a
    # time index. Default False.
    'timelapse': False,

    # (int or None) - Maximum distance in pixels an object may travel
    # between consecutive frames when linking: trackpy's search_range,
    # or btrack's max search radius. Too small fragments tracks, too
    # large causes identity swaps and SubnetOversize failures. None
    # auto-searches downward from 500 for trackpy and falls back to 100
    # for btrack. Default None.
    'timelapse_displacement': None,

    # (list) - Slice of frame indices [start, end] kept from each batch
    # before tracking, e.g. [0,10] to work on the first ten frames while
    # tuning settings. The list is ignored unless it has at least two
    # elements, which is why the shipped default [5,] has no effect.
    # Default [5,].
    'timelapse_frame_limits': [5],

    # (int) - Number of consecutive frames an object may vanish (e.g.
    # missed by segmentation) and still be re-linked to the same track
    # by trackpy. Raise it when tracks fragment because objects blink
    # out; too high risks merging two different objects into one track.
    # Not used by the btrack mode. Default 3.
    'timelapse_memory': 3,

    # (str) - Which tracker links objects between frames. 'trackastra'
    # is a transformer that tops the Cell Tracking Challenge
    # leaderboard, needs no tuning and links divisions natively;
    # 'ultrack' solves segmentation and linking as one integer program
    # and wins on densely packed or 3D data at the cost of a longer
    # solve; 'trackpy' needs a search radius and memory; 'btrack' needs
    # a motion model; 'iou' just overlaps consecutive frames and drifts
    # under fast motion. Default 'trackastra'.
    'timelapse_mode': 'trackastra',

    # (list) - Which segmented objects are tracked across frames and
    # relabelled with track IDs: any subset of ['cell', 'nucleus',
    # 'pathogen']; any other value aborts the run with a message. Each
    # extra entry costs a full additional tracking pass. Tracking nuclei
    # is often more stable than cells when cells touch. Default
    # ['cell'].
    'timelapse_objects': ['cell'],

    # (bool) - After linking, drop every track not present in all frames
    # (trackpy filter_stubs over the full stack length), keeping only
    # objects tracked from first frame to last. Enable for clean
    # per-object time courses; expect to lose cells that divide, enter
    # or leave the field, so object counts fall. Default False.
    'timelapse_remove_transient': False,

    # (str) - How Trackastra turns predicted association scores into
    # tracks: 'greedy' takes the best match per object and is fast,
    # 'ilp' solves the assignment globally and is more accurate on
    # crowded or dividing populations but needs the trackastra ilp extra
    # and considerably more time. Default 'greedy'.
    'trackastra_linking': 'greedy',

    # (str) - Which pretrained Trackastra checkpoint links the frames;
    # 'general_2d' is the all-round 2D model and covers most live-cell
    # data without retraining. Only consulted when
    # timelapse_mode='trackastra'. Change it only if you hold a
    # checkpoint trained on imaging that looks unlike yours. Default
    # 'general_2d'.
    'trackastra_model': 'general_2d',

    # (float) - Standard deviation of the Gaussian blur applied while
    # turning the segmentation labels into the contour map Ultrack
    # builds its candidate objects from. Zero keeps the boundaries
    # exactly as Cellpose drew them; one to four softens them so the
    # joint solver is free to redraw boundaries between objects that
    # were merged or split. Only consulted when
    # timelapse_mode='ultrack'. Default 0.0.
    'ultrack_contour_sigma': 0.0,

    # (float) - Cost the Ultrack solver pays to split one track into two
    # daughters; the value is negative and the more negative it is the
    # more readily divisions are accepted. Make it less negative when a
    # replication assay over-calls divisions on touching cells, more
    # negative when real division events are being missed. Only
    # consulted when timelapse_mode='ultrack'. Default -0.1.
    'ultrack_division_weight': -0.1,

    # (float) - The largest jump in pixels Ultrack will consider when
    # linking an object in one frame to a candidate in the next;
    # anything further apart is never joined, so the track breaks
    # instead. Raise it for fast-moving or sparsely sampled cells, lower
    # it on crowded fields where a generous radius invites identity
    # swaps. Only consulted when timelapse_mode='ultrack'. Default 25.0.
    'ultrack_max_distance': 25.0,

    # (int) - How many worker processes Ultrack runs during its
    # candidate-segmentation and linking passes; they all write into the
    # same temporary sqlite store, so extra workers cut wall-clock on
    # long movies but add database contention and memory. Leave it at
    # one for short batches or a busy machine. Only consulted when
    # timelapse_mode='ultrack'. Default 1.
    'ultrack_n_workers': 1,

    # (bool) - Legacy image-upscaling toggle: no code in spaCR reads
    # this key (or upscale_factor), so enabling it changes nothing about
    # image size, segmentation or measurements. Kept only for
    # settings-file compatibility. To change the working resolution use
    # the Cellpose resize / target_height / target_width settings
    # instead. Default False.
    'upscale': False,

    # (float) - Scale factor that the inactive 'upscale' toggle would
    # have applied. Nothing in spaCR reads this key, so changing it has
    # no effect on image size, segmentation or measurements; use the
    # Cellpose resize / target_height / target_width settings to change
    # resolution. Default 2.0.
    'upscale_factor': 2.0,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

    # (float or None) - Width of one pixel in micrometres in the image
    # plane, assumed square. Used with voxel_size_z_um to derive
    # anisotropy and to turn voxel counts into physical volumes and
    # surface areas. Note this is a different setting from um_per_pixel,
    # which only sizes the scale bar drawn on figures and never reaches
    # a measurement. This one does reach measurements, but only on a 3-D
    # run: a 2-D run never applies it, because doing so would turn every
    # *_area from px2 into um2 under an unchanged column name. Default
    # None.
    'voxel_size_xy_um': None,

    # (float or None) - Spacing between consecutive z planes in
    # micrometres, straight off the acquisition settings. Together with
    # voxel_size_xy_um it derives anisotropy, so setting these two is
    # the safer way to get it right, and it is also what converts object
    # volumes from voxel counts into um3. Changing it rescales every
    # physical z quantity and the anisotropy used for segmentation; it
    # has no effect on a 'project' run. Measure uses the pair to report
    # 3-D morphology in micrometres rather than voxels, and records
    # which it used in the measurement_units column. Default None.
    'voxel_size_z_um': None,

    # (int or None) - Which axis of the incoming array holds z, as 0, 1
    # or 2. None asks spaCR to work it out from the shape, which it can
    # do only when one axis is clearly shorter than the other two (a
    # 21x512x512 or 512x512x21 stack); for an ambiguous shape such as
    # 64x64x64 it stops and asks rather than guessing, because guessing
    # wrong segments a transposed volume and produces plausible
    # nonsense. Set it explicitly whenever your acquisition's shape is
    # ambiguous. Default None.
    'z_axis': None,

    # (str or None) - How z is collapsed when z_segmentation_mode is
    # 'project'. 'max' takes the brightest value down the stack and is
    # the usual choice for sparse fluorescent objects; 'mean' averages,
    # which suppresses noise but dilutes anything present in only a few
    # planes; 'sum' preserves total signal so intensity stays
    # proportional to how much of the object was in the stack;
    # 'best_focus' discards every plane but the sharpest one, which
    # beats a projection when only one plane is genuinely in focus and a
    # MIP would smear the out-of-focus haze over it. Ignored by the
    # other two modes. Default 'max'.
    'z_projection': 'max',

    # (str) - Which of the three genuinely different ways to handle z is
    # used, recorded alongside the masks because their answers are not
    # comparable. 'project' collapses the stack with z_projection and
    # segments the resulting single plane, which is what spaCR has
    # always effectively done and the only mode whose masks the Measure
    # module can consume. 'stitch' segments every plane independently in
    # 2-D and then links labels down the stack by overlap (see
    # stitch_threshold); it never computes a distance along z, so
    # anisotropy does not enter it and an object invisible in one plane
    # breaks in two. 'volumetric' segments the whole volume at once
    # using the z gradient, which finds objects that no single plane
    # shows but is acutely sensitive to a wrong anisotropy, which it
    # therefore requires. Default 'project'.
    'z_segmentation_mode': 'project',

    # (bool) - Treat each field as a z-stack instead of a flat image,
    # turning on the z_segmentation_mode / anisotropy / stitch_threshold
    # controls below. Off, spaCR runs the ordinary 2-D path and none of
    # the z code executes at all, so your masks are identical to a run
    # from before this setting existed. On, spaCR requires an array that
    # still has a z axis when it reaches segmentation and stops with an
    # error if it does not; note that the standard image ingest
    # currently collapses z into one plane per field while organising
    # the raw files, so this setting only has something to work with
    # when you feed spaCR volumes directly through the Python API.
    # Default False.
    'z_stack': False,

}

# Fill in anything left unset, then check the source path.
settings = set_default_settings_preprocess_generate_masks(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
preprocess_generate_masks(settings)

## Where the output went

A `masks/` folder beside the source images, and a measurement database seeded with the objects it found.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.